# Week 8 guided practice

CML3111. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and upload engineering_data.csv into Colab with the Files panel before running the cells. The target in this notebook is motor temperature in deg C.

## Today’s learning

*PowerPoint slide 2*

WHAT WE WILL DO
Predict motor temperature from load and ambient temperature in three ways: the average, a straight line and a decision tree. Learn the words that go with them: feature, target, training, error and loss.

HOW YOU WILL SHOW IT
You can say in plain words what each method learned and read its error in deg C. You can explain why some rows stay hidden while a model learns.

## Today’s model is supervised regression

*PowerPoint slide 3*

Supervised learning learns from examples that come with answers. Regression predicts a number.

## The supplied file

*PowerPoint slide 6*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep them beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.

## Operations for the next example

*PowerPoint slide 7*

01

Import and short names: import pandas as pd loads pandas and gives it the short name pd. pd.read_csv refers to a function in that library.

02

Reading a CSV: pd.read_csv("practice_filename.csv") reads the named file into a DataFrame: a table with labelled columns and row labels. The file must exist in the working folder.

03

Counting dimensions: equipment_data.shape gives (number of rows, number of columns). shape is a property, so it has no parentheses.

## Operations for the next example

*PowerPoint slide 8*

Column data types: equipment_data.dtypes reports each column’s stored type. int64 means integers, float64 means decimal numbers, and object often contains text. Numeric blanks commonly appear as NaN.

A preview of rows: equipment_data.head() returns the first five rows. It leaves the table unchanged. A preview is not the entire dataset.

## Reading the code syntax

*PowerPoint slide 9*

= assigns a value to a name; == compares values. Parentheses pass inputs or continue an expression across lines. print displays a value; len counts entries. int and float convert to integer and decimal numeric values.

## Loading the supplied observations

*PowerPoint slide 10*

raw_data: original CSV table; keep it unchanged so the analysis can be checked.

Run the cell and read the output below it.

In [ ]:
import pandas as pd
raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)

## Some rows stay hidden while the model learns

*PowerPoint slide 11*

Training rows teach. Validation rows compare models. Test rows are opened once, at the end.

## Rows used for modelling

*PowerPoint slide 12*

Keep the first occurrence of each record_id. Retain recorded targets and load_pct within 0–100. Then separate E1–E4 for training, E5 for validation and E6 for the final test_data.

Fill missing input values using training data only. Keep final-test targets out of model selection.

## Operations for the next example

*PowerPoint slide 13*

01

Reading chained operations: Read a chain from left to right. In equipment_data["target"].mean(), select the target Series first, then calculate its mean. Each step uses the previous step’s result.

02

A separate working table: equipment_data.copy() creates a working copy. Assigning columns on this copy preserves the source table, so you can compare before and after.

03

Finding recorded cells: notna() returns True where a value is present. Use it to retain observations with a recorded target before modelling.

## Operations for the next example

*PowerPoint slide 14*

01

An inclusive range: series.between(0, 100) is True for values from 0 through 100, including both limits. The resulting mask helps select valid rows.

02

Selecting columns: equipment_data["column"] returns one labelled column, a Series. equipment_data[["a", "b"]] returns a DataFrame containing those named columns in that order.

03

Selecting rows with a mask: A column comparison produces one True or False per row. equipment_data[mask] keeps the True rows. & combines masks: both conditions must hold. Parenthesise each comparison.

## Operations for the next example

*PowerPoint slide 15*

Keeping the first observation: equipment_data.drop_duplicates("record_id") returns rows with the first occurrence of each ID, preserving their order. It does not overwrite the source unless you assign the result.

Membership in a group list: series.isin(["E1", "E2"]) returns True for entries matching either listed label. equipment_data[that_mask] keeps all observations belonging to those groups.

## Prepare valid observations

*PowerPoint slide 16*

First keep the first record ID.

Then ask two questions for each remaining row: is the target recorded, and is load between 0 and 100?

Only a row passing both checks stays.

Missing inputs are handled later using training data only.

## Preparing the supplied modelling rows: step 1

*PowerPoint slide 17*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()
target_is_recorded = modelling_data["target"].notna()
load_is_valid = modelling_data["load_pct"].between(0, 100)
rows_to_keep = target_is_recorded & load_is_valid
print(target_is_recorded.head())
print(load_is_valid.head())
modelling_data = modelling_data[rows_to_keep].copy()
print(len(raw_data), len(modelling_data))

## Choose groups before choosing rows

*PowerPoint slide 18*

isin(list_of_labels) answers True when a row’s equipment ID appears in the list.

Here E1–E4 supply training rows, E5 validation rows and E6 test rows.

The groups stay separate so one equipment group cannot appear on both sides.

## Preparing the supplied modelling rows: step 2

*PowerPoint slide 19*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
is_training_equipment = modelling_data["equipment_id"].isin(
    ["E1", "E2", "E3", "E4"]
)
is_validation_equipment = modelling_data["equipment_id"].isin(
    ["E5"]
)
is_test_equipment = modelling_data["equipment_id"].isin(["E6"])

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 20*

training_data: rows used to learn model settings; only these rows may determine imputation values.

validation_data: separate rows used to compare models without fitting on them.

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 21*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
training_data = modelling_data[is_training_equipment].copy()
validation_data = modelling_data[is_validation_equipment].copy()
test_data = modelling_data[is_test_equipment].copy()
feature_columns = ["load_pct", "ambient_c"]
print(len(training_data), len(validation_data), len(test_data))

## A line through the points beats the average

*PowerPoint slide 22*

Higher load goes with higher motor temperature. The line follows the dots; the dashed average ignores load.

## Four misses, measured three ways

*PowerPoint slide 23*

MAE  = mean of the absolute errors = 2.75 deg C
MSE  = mean of the squared errors = 12.25 (deg C squared)
RMSE = square root of the MSE = 3.5 deg C
These four readings are invented to show the arithmetic.

## Fitting a line to observations

*PowerPoint slide 24*

The fitted line minimises the sum of squared vertical errors.

## The fitted line is an equation

*PowerPoint slide 25*

predicted motor temperature = 31.2 + 0.666 × load_pct

A fit on load alone, to show the idea.
Intercept 31.2: the prediction when load is 0 percent.
Slope 0.666: each extra 1 percent of load adds 0.666 deg C to the prediction.
At 70 percent load the line predicts 77.8 deg C.
With both inputs the model learns one slope for each: 1.66 + 0.695 × load_pct + 1.054 × ambient_c. scikit-learn keeps these in intercept_ and coef_.

## scikit-learn reads X as a table and y as a column

*PowerPoint slide 26*

X has one row per observation and one column per feature. y holds one answer per row.

## NumPy arrays and their shape

*PowerPoint slide 27*

np.array turns a list of numbers into an array.
A list inside a list gives rows and columns.
shape gives (rows, columns). One-dimensional y shows (rows,).
Later, a model's predict returns an array like this. Call .tolist() to get a plain Python list.

X has shape (3, 2) and y has shape (3,). Every row of X lines up with one entry of y.

In [ ]:
import numpy as np
X = np.array([[59.15, 31.71],
              [77.46, 29.97],
              [69.6, 19.87]])
y = np.array([76.847, 87.047, 71.256])
print(X.shape)
print(y.shape)
print(round(y.mean(), 3))

## Training walks downhill on the loss

*PowerPoint slide 28*

Loss
One number for how wrong the model is. We use the mean squared error: square each miss, then average.
Gradient
Which way is downhill, and how steeply.
Learning rate
The size of each step. Too big overshoots; too small takes too long.
Gradient descent
Guess, measure the loss, step downhill, repeat.

Each orange dot is one try at the slope. The loss falls as the slope approaches the best value.

## Gradient descent in a few lines of NumPy

*PowerPoint slide 29*

The best slope is 2, because y is always twice x.
w starts at 0. Each step moves it a little toward 2.
learning_rate sets the size of each step. np.mean averages an array. ** is the power from Week 3.
The loss falls on every step.

LinearRegression finds its line with a direct formula. Gradient descent is the walk behind neural networks, which we train in Semester 2.

In [ ]:
import numpy as np
x = np.array([1, 2, 3])
y = np.array([2, 4, 6])
w, learning_rate = 0.0, 0.05
print("step w loss")
for step in range(1, 6):
    gradient = 2 * np.mean(x * (w * x - y))
    w = w - learning_rate * gradient
    loss = np.mean((w * x - y) ** 2)
    print(step, round(w, 3), round(loss, 3))

**Example output**

```text
step w loss
1 0.933 5.31
2 1.431 1.51
3 1.697 0.43
4 1.838 0.122
5 1.914 0.035
```

## A model cannot learn from a blank, so we fill it first

*PowerPoint slide 30*

An imputer fills each blank with the median of the training rows. Median means the middle value.

## A pipeline runs its steps in order

*PowerPoint slide 31*

A pipeline is a short to-do list. Write it once, then call fit and predict on the whole list.

## Training a model with missing inputs

*PowerPoint slide 32*

training_data[feature_columns] selects the two input columns as a table. training_data["target"] selects the observed target as a Series.

SimpleImputer(strategy="median") learns input fill values from training rows. make_pipeline links that operation to LinearRegression. fit learns the steps and predict reuses them.

## Operations for the next example

*PowerPoint slide 33*

Returning plain values: series.tolist() or array.tolist() returns a Python list of values, without pandas row labels. Preserve their order when pairing actual values and validation_predictions.

Rounded display: series.round(3) returns values rounded to three decimal places. Python round(number, 3) rounds one number. Keep unrounded calculations for returned assessment answers.

## A sequence of model steps / Filling model inputs

*PowerPoint slide 34*

make_pipeline(imputer, estimator) links preprocessing and prediction. fit learns the imputer and estimator on training rows. predict applies the learned steps to new rows in the same order.

SimpleImputer(strategy="median") learns a median for each input column during fit. It uses those training medians to fill gaps later. Never learn replacement values from validation or test rows.

## Creating linear regression / Learning from training examples

*PowerPoint slide 35*

LinearRegression() creates an unfitted model. It will learn coefficients and an intercept from training examples. Creating it alone does not train it or make predictions.

model.fit(input_table, observed_targets) learns model settings from paired rows. The two inputs must describe the same observations in the same order. Call fit before predict.

## Predicting new targets / Selecting a short preview

*PowerPoint slide 36*

model.predict(input_table) returns one prediction per input row. Use the same feature columns and order as during training. Prediction uses learned settings; it does not refit the model.

values[:5] selects positions 0 through 4, or fewer if the sequence is shorter. The stop position is excluded. This preview does not change the full array stored in the variable.

## Standard names for model inputs

*PowerPoint slide 37*

01

X_train: training input columns. y_train: the matching observed targets.

02

X_valid: validation input columns. y_valid: their observed targets, kept for evaluation.

03

Capital X usually means a two-dimensional table; lowercase y usually means one target per row.

## Separate inputs from answers

*PowerPoint slide 38*

X_train contains the input columns; y_train contains their observed targets.

X_valid contains the same input columns for validation; y_valid holds the answers for checking predictions.

Never include target among the inputs.

## Training and predicting on the supplied data: step 1

*PowerPoint slide 39*

X_train: input columns used for training.

Run the cell and read the output below it.

In [ ]:
X_train = training_data[feature_columns]
y_train = training_data["target"]
X_valid = validation_data[feature_columns]
y_valid = validation_data["target"]
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

## Create, then train, then predict

*PowerPoint slide 40*

Creating an estimator sets up an empty model. fit learns settings from training inputs and their targets. predict uses those learned settings on new inputs; it does not learn from the validation targets.

## Training and predicting on the supplied data: step 2

*PowerPoint slide 41*

X_train: input columns used for training.

Run the cell and read the output below it.

In [ ]:
linear_model = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression()
)
linear_model.fit(X_train, y_train)
validation_predictions = linear_model.predict(X_valid)
print(validation_predictions[:5].round(3).tolist())

## A decision tree asks yes or no questions

*PowerPoint slide 42*

max_depth is a hyperparameter, a setting you choose before training. It limits how many questions in a row.

## Baseline and tree comparisons

*PowerPoint slide 43*

A mean baseline predicts the training target mean for every validation observation. A regression tree splits input space and predicts from training observations in each leaf.

Use the same training rows and validation rows for both. max_depth=4 limits tree depth and random_state=17 makes the example reproducible.

## A training-mean baseline / Creating a regression tree

*PowerPoint slide 44*

DummyRegressor(strategy="mean") learns the mean training target during fit. It predicts that same value for every new row. Compare a more complex model against this simple baseline.

DecisionTreeRegressor splits inputs into regions with different predicted values. max_depth limits the number of split levels. random_state fixes random choices so the example can be repeated.

## Start with a simple reference

*PowerPoint slide 45*

A training-mean baseline predicts the same value for every validation row. fit calculates that value from y_train.

Compare its error with the more complex model on the same validation rows.

## Baseline and tree on the same CSV rows: step 1

*PowerPoint slide 46*

baseline_model: a simple reference predictor which always predicts the training target mean.

Run the cell and read the output below it.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

## Compare a second kind of model

*PowerPoint slide 47*

A regression tree learns splits in the input columns. max_depth limits its split levels.

The imputer fills missing inputs using training medians; the tree then learns from those filled inputs.

## Baseline and tree on the same CSV rows: step 2

*PowerPoint slide 48*

baseline_model: a simple reference predictor which always predicts the training target mean.

Run the cell and read the output below it.

In [ ]:
tree_model = make_pipeline(
    SimpleImputer(strategy="median"),
    DecisionTreeRegressor(max_depth=4, random_state=17),
)
tree_model.fit(X_train, y_train)
print(baseline_model.predict(X_valid)[:5].round(3).tolist())
print(tree_model.predict(X_valid)[:5].round(3).tolist())

## Matching actual values and predictions

*PowerPoint slide 49*

Every validation row has an actual target and its corresponding prediction. Subtract in the same order. abs() gives error magnitude.

MAE averages absolute errors. RMSE squares errors, averages them, then takes the square root. Both have the target’s unit.

## Operations for the next example

*PowerPoint slide 50*

01

Adding or replacing a column: equipment_data["name"] = values assigns that column. An array follows row order. A Series aligns by row label, so keep labels paired with the intended observations.

02

Column arithmetic: Arithmetic on columns calculates one result per row. Series align by row label. Divide only where the denominator is valid and keep actual and predicted observations paired.

03

Mean of recorded values: series.mean() adds known values and divides by their count. pandas skips missing values by default. The result has the same unit as the original column.

## Operations for the next example

*PowerPoint slide 51*

Magnitude of each error: series.abs() makes negative values positive while leaving nonnegative values unchanged. Averaging these magnitudes gives mean absolute error.

## Squared error and square root

*PowerPoint slide 52*

error ** 2 squares each error. After averaging, ** 0.5 takes the square root. This gives root mean squared error, which penalises large errors more strongly than mean absolute error.

## Line up observations and predictions

*PowerPoint slide 53*

Copy record IDs and observed targets from validation rows.

Add predictions in that same order.

Each row now pairs one observation with its own predicted value.

## Errors from the supplied validation observations: step 1

*PowerPoint slide 54*

validation_results: actual and predicted targets paired with each record ID.

Run the cell and read the output below it.

In [ ]:
validation_results = validation_data[
    ["record_id", "target"]
].copy()
validation_results["predicted"] = validation_predictions

## Measure how far predictions missed

*PowerPoint slide 55*

Subtract predicted from observed values.

Positive errors mean underprediction; negative errors mean overprediction. abs removes signs for MAE.

Squaring before averaging and taking a square root gives RMSE.

## Errors from the supplied validation observations: step 2

*PowerPoint slide 56*

validation_results: actual and predicted targets paired with each record ID.

Run the cell and read the output below it.

In [ ]:
prediction_errors = (
    validation_results["target"] - validation_results["predicted"]
)
mae = prediction_errors.abs().mean()
rmse = (prediction_errors**2).mean() ** 0.5
print(validation_results.head())
print(round(float(mae), 4), round(float(rmse), 4))

## One MAE for each model on the validation rows

*PowerPoint slide 57*

A dictionary pairs each name with a model we fitted earlier.
items() gives one name and one model per turn of the loop.
mean_absolute_error(actual, predicted) is the MAE from earlier slides.

The numbers are in deg C. Each model predicts the same validation rows.

In [ ]:
from sklearn.metrics import mean_absolute_error
models = {
    "baseline": baseline_model,
    "line": linear_model,
    "tree": tree_model,
}
for name, model in models.items():
    predicted = model.predict(X_valid)
    error = mean_absolute_error(y_valid, predicted)
    print(name, round(error, 2))

**Example output**

```text
baseline 14.85
line 0.78
tree 4.79
```

## Three models on the validation rows

*PowerPoint slide 58*

The line misses by about 0.78 deg C on average. The average guess misses by 14.85 deg C.
All three models used the same training rows and the same validation rows.

## Pandas inside the required function

*PowerPoint slide 59*

The assignment caller supplies equipment_records. pd.DataFrame(records) gives those records named columns. Calculate with pandas, then return the published Python type.

to_dict("records") makes a list of row dictionaries. Series.tolist() makes a list of values. A grouped Series.to_dict() maps group labels to their values.

## Operations for the next example

*PowerPoint slide 60*

01

Returning a dictionary: series.to_dict() maps Series labels to values. equipment_data.to_dict("records") instead returns a list of row dictionaries. Choose the form required by the assignment.

02

A table from Python values: pd.DataFrame(records) builds a table from row dictionaries. With a dictionary of column lists, keys become column names and each list supplies that column’s values.

03

Grouping rows: equipment_data.groupby("label")["value"] groups the value column by matching labels. Apply mean() or sum() to calculate one result per group. Missing group labels are excluded by default.

## Defining and calling a function

*PowerPoint slide 61*

def gives a reusable block a name and input parameters. The indented statements run when it is called. return sends a result back to the caller; print displays a result but does not return it.

## A function called with the supplied records

*PowerPoint slide 62*

equipment_records: the input list of row dictionaries supplied to the function.

Run the cell and read the output below it.

In [ ]:
def mean_load_by_equipment(equipment_records):
    equipment_data = pd.DataFrame(equipment_records)
    mean_load_by_equipment_group = equipment_data.groupby(
        "equipment_id"
    )["load_pct"].mean()
    return mean_load_by_equipment_group.to_dict()
equipment_records = modelling_data.to_dict("records")
print(mean_load_by_equipment(equipment_records))

## Your issued task

*PowerPoint slide 64*

01

Investigate motor winding temperature.

02

Target units: deg C.

03

These are synthetic teaching records for practice.

04

Each record is one operating observation; equipment_id repeats across observations.

## predict_regression(training_records, feature_records)

*PowerPoint slide 65*

Fit a training-mean baseline, linear regression and a decision tree (max_depth=4, random_state=17) using load_pct and ambient_c only.

Return three prediction lists under baseline, linear and tree.

Feature records have no target.

Fit preprocessing only on training data.

Keep the exact names and outputs. Check types, units and the issued brief.

## regression_errors(actual, predicted)

*PowerPoint slide 66*

Return mae and rmse, calculated over the same paired observations, with full precision.

Keep the exact names and outputs. Check types, units and the issued brief.

## Written evidence

*PowerPoint slide 67*

01

Use the cell headed “My written evidence”.

02

Explain the supplied scenario in your own words.

03

State target units, reasons and limitations.

04

Submit one complete notebook containing code and written evidence.